# Q-BenchMed in Colab

Does quantum optimisation help on a real biomedical decision problem?

This notebook runs the comparison the project exists for. Given a set of
biomarkers you could acquire and a set of clinical outcomes a rule engine
can reach, which biomarkers cover the most outcomes? Classical solvers and
QAOA receive the **identical QUBO**, so any difference between them is the
optimiser rather than the problem.

Three things happen below:

1. every solver on one QUBO, at a size a state-vector simulator can hold;
2. the real 66-biomarker instance, where greedy selection stalls;
3. the measurement that explains the quantum result, read from the shipped
   result tables.

About four minutes end to end on a free Colab CPU runtime. No GPU, no
quantum provider account, and nothing to configure.

**Nothing here has run on quantum hardware, and no quantum advantage is
claimed.** The quantum side is simulated, and simulation stops at sixteen
inputs.

## Setup

Clones the repository and installs it. Takes a minute or so, mostly pip.

In [ ]:
!git clone --depth 1 https://github.com/nabvian/qbenchmed.git
%cd qbenchmed
!pip install -q -e .

import qbm
print('Q-BenchMed ready')

## 1. Every solver, one QUBO

One instance is generated, one QUBO is built from it, and every method
below minimises *that object* — not a re-derivation of it. Exhaustive
enumeration of the same QUBO supplies the certified optimum every other
answer is scored against.

Greedy is the exception, and is labelled `native`: it works on the coverage
structure rather than on a QUBO. It is placed on the QUBO axis by holding
its selection fixed and completing the auxiliary variables optimally, which
changes nothing about what greedy does and makes its answer comparable. The
report never averages it with the QUBO solvers.

In [ ]:
import time
import pandas as pd
from qbm import instances as ins, runner as rn

N_INPUTS = 12          # raise toward 16 if you want a harder instance
SEED = 0

inst = ins.generate('degree_capped', N_INPUTS,
                    round(N_INPUTS * 88 / 66), seed=SEED)
K = max(2, round(0.30 * N_INPUTS))     # at most K biomarkers in the panel

spec = rn.RunSpec(benchmark_id='QBM-COLAB', input_budget=K,
                  encoding='pairwise', notes='colab demo')

plan = [{'algorithm': 'qubo_exhaustive'},
        {'algorithm': 'greedy'},
        {'algorithm': 'simulated_annealing', 'seed': SEED},
        {'algorithm': 'tabu', 'seed': SEED}]
plan += [{'algorithm': 'qaoa', 'depth': p, 'shots': 4096,
          'seed': SEED, 'n_restarts': 3} for p in (1, 2, 3)]

t = time.perf_counter()
out = rn.run_comparison(inst, spec, plan)
print(f'{time.perf_counter() - t:.1f}s')

pd.DataFrame([{
    'method': f'qaoa p={r.qaoa_depth}' if r.algorithm == 'qaoa' else r.algorithm,
    'formulation': r.formulation,
    'qubo energy': round(r.qubo_energy, 4),
    'gap to optimum': round(r.qubo_gap_abs, 4),
    'found optimum': abs(r.qubo_gap_abs) < 1e-9,
    'coverage': round(r.coverage_fraction, 4),
} for r in out])

One instance is one instance. The published rates come from twenty-five —
five sizes from eight to sixteen inputs, five seeds each — and QAOA at depth
two reaches the certified optimum in 92% of them, above simulated annealing
at 88% and greedy at 80%, below tabu at 100%.

With twenty-five instances per configuration those rates are not separated
by the data: 92% carries a 95% interval of [77%, 98%]. **Comparable, not
better** is the honest reading.

To reproduce the full table, run `python experiments/run_headtohead.py`.
It takes about seven and a half minutes.

## 2. The real instance, where greedy stalls

`QBMED-HEME-001` is 66 biomarkers and 88 outcomes, taken from an audit
export of a working pathology engine's trigger matrix. It contains no
patient data.

What makes it worth solving is that it is **conjunctive**: most of its rule
arms fire only when several biomarkers are present together. Coverage is
then not submodular, and marginal-gain selection has no guarantee left.

This instance is far too large to simulate — its exact encoding needs 154 or
more qubits — so it is solved classically, by a certified branch-and-bound.

In [ ]:
from qbm import classical as cl

heme = ins.heme_benchmark()
conditional = [a for a in heme.arms if len(a[1]) > 0]
conjunctive = [a for a in conditional if len(a[1]) > 1]
print(f'{heme.instance_id}: {heme.n_inputs} biomarkers, {heme.n_outcomes} outcomes')
print(f'{len(conjunctive)} of {len(conditional)} rule arms need more than one '
      f'biomarker present at once')

rows = []
for budget in (5, 10, 20, 30, 33, 37, 45, 66):
    certified = cl.ilp(heme.A, heme.w, heme.c, budget, arms=heme.arms)
    greedy = cl.greedy(heme.A, heme.w, heme.c, budget, arms=heme.arms)
    rows.append({'panel size (at most)': budget,
                 'certified': f'{certified.coverage_fraction:.2%}',
                 'greedy': f'{greedy.coverage_fraction:.2%}'})

pd.DataFrame(rows)

Greedy climbs to 94.32% and then stops, and stays there however much budget
it is given. The certified solver keeps going to 96.59%.

The reason is structural. The outcomes greedy cannot reach fire only when
several biomarkers are present *together*, so no single biomarker shows a
positive marginal gain and greedy has nowhere to step. This is exactly the
failure mode submodularity rules out, and it is visible here only because
the instance is conjunctive.

96.59% is also the ceiling. Three outcomes cannot be reached by any
selection at any budget, because their arms are never jointly satisfiable
within these 66 biomarkers. Ask for a 100% floor and the solver refuses
rather than quietly returning its best effort:

In [ ]:
best = cl.ilp(heme.A, heme.w, heme.c, heme.n_inputs, arms=heme.arms)
unreachable = heme.n_outcomes - len(best.covered_outcomes)
print(f'{unreachable} of {heme.n_outcomes} outcomes are unreachable at any budget')
print(f'ceiling: {best.coverage_fraction:.2%}')

## 3. Why QAOA does not do better here

This result is a property of the encoding rather than of any solver, which
is why it survived a correction that changed the headline.

The budget constraint — at most K biomarkers — is expressed as a penalty
term added to the objective. Measured on the same instances, those penalty
terms run far larger than the coverage signal, and the energy band holding
anything useful is a sliver of the spectrum. The measurement ships with the
repository:

In [ ]:
landscape = pd.read_csv('results/penalty_landscape.csv')
landscape['useful_band_pct'] = 100 * landscape.useful_span / landscape.span

landscape.groupby('n_inputs').agg(
    penalty_vs_signal=('penalty_dominance', 'mean'),
    useful_band_pct=('useful_band_pct', 'mean'),
    spectrum_span=('span', 'mean'),
).round(3)

The penalty terms exceed the coverage signal by 85 to 374 times, and the
useful band shrinks as the problem grows — from 0.91% of the spectrum at
eight inputs to 0.24% at sixteen.

QAOA spends its variational capacity climbing out of the penalty bulk
rather than discriminating among good solutions. That was tested rather
than asserted: scaling the penalty coefficient down by more than an order
of magnitude left the optimum probability in the same range, and below a
threshold the QUBO's own minimiser started violating the budget, at which
point it no longer encodes the problem.

## 4. Remove the penalty

If the penalty is the problem, an ansatz that never leaves the valid panels
should do better. This one starts from the Dicke state — the equal
superposition of every panel with exactly K biomarkers — mixes with an XY
ring that conserves how many are selected, and uses coverage alone as its
cost. No penalty, no slack qubits.

The fair baseline is a random valid panel, the Dicke state before any layer.
Probability of sampling the optimum, same instance as section 1:

In [ ]:
from qbm import constrained as cn
from qbm import qubo as qb
from qbm.qaoa import Qaoa

diag = cn.coverage_diagonal(inst.A, inst.w)
opt, _ = cn.optimum_at_weight(diag, K)
p0 = cn.dicke_state(N_INPUTS, K).probabilities()
random_p = p0[diag.reshape(-1) <= opt + 1e-9].sum()

qp = qb.build_max_coverage(inst.A, inst.w, inst.c, K,
                           encoding='pairwise', arms=inst.arms)
qopt = qp.all_energies()[1].min()

rows = [{'ansatz': 'random valid panel', 'qubits': N_INPUTS, 'P(optimum)': random_p}]
for p in (1, 2):
    pen = Qaoa(qp).run(p, shots=4096, seed=SEED, n_restarts=3, optimum_energy=qopt)
    con = cn.ConstrainedQaoa(inst.A, inst.w, K).run(p, shots=4096, seed=SEED,
                                                    n_restarts=3, optimum_energy=opt)
    rows += [{'ansatz': f'penalty p={p}', 'qubits': qp.n_vars,
              'P(optimum)': pen.optimum_probability},
             {'ansatz': f'constrained p={p}', 'qubits': N_INPUTS,
              'P(optimum)': con.optimum_probability}]

df = pd.DataFrame(rows)
df['vs random'] = (df['P(optimum)'] / random_p).round(2)
df.round(4)

On the 25 instances of the published head-to-head, the constrained ansatz
beats a random valid panel every time — a median 9× at depth 2 — and the
penalty version almost always does worse than random guessing. This is noiseless
simulation: preparing the Dicke state costs two-qubit gates, and whether the
gain survives real noise is untested. Section 8 of the report has the full
comparison, including slices of the real instance where only the constrained
version fits in a simulator.

## 5. A defect worth knowing about

An earlier revision of this repository reported that classical heuristics
beat QAOA at every depth. That was an artefact of the parameter search.

The variational parameter `gamma` was drawn from `[0, pi)` and documented as
one period. For this spectrum the period is `[0, 2 pi)`, and on a probe
instance the best value sat at 6.07 — outside the interval the optimiser
ever started in. Compounding it, the penalised spectrum spans thousands of
energy units, which puts hundreds of local minima inside one period, so
three random restarts of a local optimiser was a lottery.

The search now sweeps a coarse grid over one computed period and refines
from its best points. QAOA at depth two went from 72% to 92%, and the
headline conclusion reversed.

You can see the period the search derives for your own instance:

In [ ]:
from qbm import qubo as qb
from qbm.qaoa import Qaoa

q = qb.build_max_coverage(inst.A, inst.w, inst.c, K,
                          encoding='pairwise', arms=inst.arms)
print(f'gamma period for this instance: [0, {Qaoa(q).gamma_period():.4f})')
print('the old search drew from [0, 3.1416)')

Section 7 of [`reproducibility/REPORT.md`](reproducibility/REPORT.md)
records this alongside two earlier defects, one of which invalidated a
whole noise sweep. The invalid table is kept in `results/` rather than
deleted.

## Scope

A covering panel is a solution to a coverage objective over a rule graph.
It does not follow that any biomarker is clinically unnecessary, that a
smaller panel is safe, or that the outcome set is clinically validated
because it can be optimised. Q-BenchMed is not a clinical validator, a
diagnostic device, or a regulatory assessment.

Nothing here extrapolates beyond sixteen inputs, which is where
state-vector simulation stops.

---

## Where to go next

- [`RESULTS.md`](RESULTS.md) — everything measured so far, in one page.
- [`USAGE.md`](USAGE.md) — running the Rust platform on a project of your own.
- [`docs/formulation.md`](docs/formulation.md) — the mathematics.
- [`BENCHMARK_INVARIANTS.md`](BENCHMARK_INVARIANTS.md) — the seven rules the
  benchmark does not break.

Licensed under AGPL-3.0-or-later.